# Memory AI Lab — 06 : Memory Store SQLite

**GPU non requis** — tourne entièrement en CPU.

## Objectif

Charger les épisodes segmentés (pipeline V4 B+C, ARI 0.90) dans la base
SQLite Ivias — fondation de la Phase B.

```
group_anon.txt + group_gold_tune.json
        ↓
Pipeline V4 B+C → épisodes segmentés
        ↓
memory_store_sqlite.py → memory.db
        ↓
Vérification : stats, search, Memory Context Pack
```

## Données requises sur Drive (`memory_ai_data/`)
```
group_anon.txt
group_embeddings_me5.npy
group_gold_tune.json          (optionnel — pour labels gold)
```

Sortie : `memory_ai_data/memory.db`

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
# sqlite-vec (optionnel — accélère la recherche vectorielle)
!pip install sqlite-vec -q 2>/dev/null || echo 'sqlite-vec non disponible — fallback cosine JS'
print('✓ OK')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale ────────────────────────────────
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

for fname in ['group_anon.txt', 'group_embeddings_me5.npy', 'group_gold_tune.json']:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Copié : {fname} ✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  ⚠️  {fname} absent sur Drive')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Charger artefacts + embeddings ────────────────────────────
import numpy as np
from pathlib import Path
from parsers.whatsapp_parser import parse_whatsapp_chat

# Artefacts
all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
print(f'Artefacts parsés : {len(all_artifacts)}')

# Embeddings mE5-base (768d)
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'
embeddings  = np.load(EMBED_CACHE)
print(f'Embeddings       : {embeddings.shape}')

assert len(all_artifacts) == len(embeddings), \
    f'Mismatch : {len(all_artifacts)} artefacts vs {embeddings.shape[0]} embeddings'

In [ ]:
# ── CELLULE 5 : Segmentation pipeline V4 B+C ──────────────────────────────
# Reproduit la segmentation utilisée pour ARI 0.90
import torch
from episode_segmenter_hybrid import HybridEpisodeSegmenter

# Paramètres optimaux V4 B+C
PARAMS = dict(
    attach_threshold  = 0.434,
    ema_alpha         = 0.787,
    time_threshold    = 330,
    boundary_k        = 0.107,
    alpha             = 0.45,
    beta              = 0.25,
    gamma             = 0.10,
    delta             = 0.20,
    rho               = 0.05,
    dormancy          = 1440,
    hard_break        = 0,
    active_penalty    = 24,
    allow_reactivation= True,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

segmenter = HybridEpisodeSegmenter(
    boundary_model_path=f'{CODE_DIR}/models/boundary_detector.pt',
    device=device,
    **PARAMS,
)

episodes = segmenter.segment(all_artifacts, embeddings)
print(f'\n✓ {len(episodes)} épisodes segmentés')
print(f'  Taille médiane  : {np.median([len(ep.artifact_indices) for ep in episodes]):.0f} msgs')
print(f'  Taille max      : {max(len(ep.artifact_indices) for ep in episodes)} msgs')

In [ ]:
# ── CELLULE 6 : Initialiser le MemoryStoreSQLite ──────────────────────────
from memory_store_sqlite import (
    MemoryStoreSQLite, NormalizedMessage, EpisodeStatus
)
from datetime import timezone

DB_PATH = f'{DATA_DIR}/memory.db'

# Supprimer si existant (rebuild propre)
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print('  Base existante supprimée')

store = MemoryStoreSQLite(
    db_path        = DB_PATH,
    centroid_alpha = 0.85,
    seuil_bas      = 0.35,
    seuil_haut     = 0.65,
    embedding_dim  = 768,
)
print(f'✓ Store initialisé : {DB_PATH}')
print(f'  sqlite-vec disponible : {store._vec_available}')

In [ ]:
# ── CELLULE 7 : Charger les épisodes dans la base ─────────────────────────
import uuid
from tqdm.auto import tqdm

episode_id_map = {}  # index épisode → episode_id SQLite

for ep_idx, ep in enumerate(tqdm(episodes, desc='Ingestion épisodes')):
    if not ep.artifact_indices:
        continue

    # Calculer le centroïd EMA sur tous les messages de l'épisode
    ep_embeddings = embeddings[ep.artifact_indices]
    centroid = ep_embeddings.mean(axis=0).astype(np.float32)

    # Premier message → créer l'épisode manuellement
    first_art = all_artifacts[ep.artifact_indices[0]]
    last_art  = all_artifacts[ep.artifact_indices[-1]]

    episode_id = store._new_episode_id()
    episode_id_map[ep_idx] = episode_id

    # Insérer l'épisode directement (bypass fast path — données historiques)
    channels     = list({a.source for a in [all_artifacts[i] for i in ep.artifact_indices]})
    participants = list({a.author for a in [all_artifacts[i] for i in ep.artifact_indices]
                         if a.author})

    ts_start = int(first_art.timestamp.timestamp()) if first_art.timestamp else 0
    ts_end   = int(last_art.timestamp.timestamp())  if last_art.timestamp  else 0

    import json
    store._conn.execute("""
        INSERT INTO episodes
          (episode_id, status, start_time, end_time, message_count,
           channels, participants, centroid_emb)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        episode_id,
        EpisodeStatus.PENDING_ENRICHMENT.value,
        ts_start, ts_end,
        len(ep.artifact_indices),
        json.dumps(channels),
        json.dumps(participants),
        store._emb_to_blob(centroid),
    ))

    # Insérer les messages
    for msg_idx in ep.artifact_indices:
        art = all_artifacts[msg_idx]
        emb = embeddings[msg_idx]
        ts  = int(art.timestamp.timestamp()) if art.timestamp else ts_start

        store._conn.execute("""
            INSERT OR REPLACE INTO messages
              (message_id, episode_id, content, author_id, timestamp,
               channel, direction, conversation_id, embedding)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            f'msg_{msg_idx:06d}',
            episode_id,
            art.content or '',
            art.author  or 'unknown',
            ts,
            art.source or 'whatsapp',
            'inbound',
            episode_id,  # conversation_id = episode_id pour données historiques
            store._emb_to_blob(emb),
        ))

store._conn.commit()
print(f'\n✓ {len(episode_id_map)} épisodes chargés dans memory.db')

In [ ]:
# ── CELLULE 8 : Stats de la base ──────────────────────────────────────────
stats = store.stats()
print('=== MemoryStoreSQLite stats ===')
print(f'  Total épisodes  : {stats["total_episodes"]}')
print(f'  Total messages  : {stats["total_messages"]}')
print(f'  Taille DB       : {stats["db_size_mb"]:.2f} MB')
print(f'  sqlite-vec      : {stats["sqlite_vec"]}')
print()
print('  Par statut :')
for status, n in sorted(stats['episodes_by_status'].items()):
    print(f'    {status:25s} : {n}')

In [ ]:
# ── CELLULE 9 : Test recherche — Memory Context Pack ──────────────────────
# Simule ce que le hook before_agent_start fait avant de répondre

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-base')

test_queries = [
    "qu'est-ce qu'on a décidé sur le budget ?",
    "organisation du voyage",
    "problème urgent à régler",
]

for query in test_queries:
    query_emb = model.encode(f'query: {query}', convert_to_numpy=True)
    pack = store.search(query_emb, query_text=query, top_k=3)

    print(f'\n🔍 Requête : "{query}"')
    print('─' * 60)
    if pack.episodes:
        print(pack.formatted)
    else:
        print('  Aucun épisode trouvé')

In [ ]:
# ── CELLULE 10 : Test decay Ebbinghaus ────────────────────────────────────
decay_stats = store.apply_decay(
    lambda_base         = 0.05,  # conservateur pour données historiques
    dormancy_threshold  = 0.3,
    archive_threshold_days = 180,
)

print('=== Decay Ebbinghaus ===')
print(f'  Épisodes traités  : {decay_stats["updated"]}')
print(f'  → DORMANT         : {decay_stats["dormant"]}')
print(f'  → ARCHIVED        : {decay_stats["archived"]}')
print()
stats2 = store.stats()
print('  Statuts après decay :')
for status, n in sorted(stats2['episodes_by_status'].items()):
    print(f'    {status:25s} : {n}')

In [ ]:
# ── CELLULE 11 : Sauvegarde sur Drive ─────────────────────────────────────
import shutil
from pathlib import Path

DRIVE_DB = f'{DRIVE_DIR}/memory.db'

store.close()

shutil.copy2(DB_PATH, DRIVE_DB)
db_size = Path(DB_PATH).stat().st_size / 1e6
print(f'✓ memory.db sauvegardé → Drive')
print(f'  Taille : {db_size:.2f} MB')
print(f'  Path   : {DRIVE_DB}')

## Étapes suivantes

**Phase B — Plugin OpenClaw :**
```typescript
// Le hook before_agent_start utilise store.search()
// pour injecter le Memory Context Pack dans le prompt
on('before_agent_start', async (ctx) => {
  const pack = await fetch(`${IVIAS_URL}/memory/search`, {
    body: JSON.stringify({ query: ctx.message, top_k: 5 })
  })
  ctx.injectContext(pack.formatted)
})
```

**Enrichissement (slow path) :**
```python
# Pour chaque épisode PENDING_ENRICHMENT :
store.enrich_episode(
    episode_id      = ep_id,
    title           = llm_title,
    summary         = llm_summary,
    goal            = llm_goal,
    decisions       = llm_decisions,
    axis_resolution = deberta_scores[0],
    axis_salience   = deberta_scores[1],
    axis_temporality= deberta_scores[2],
    model_name      = 'claude-opus-4-5',
)
```

**En attente d'enrichissement :**
```python
pending = store.pending_enrichment(limit=50)
print(f'{len(pending)} épisodes à enrichir')
```